In [26]:
import requests
import pandas as pd

In [28]:
BASE = "https://api.worldbank.org/v2"

def fetch_wb(indicator: str, start_year=2000, end_year=2023) -> pd.DataFrame:
    """Fetch indicator for all countries (country-year) with pagination."""
    url = f"{BASE}/country/all/indicator/{indicator}"
    params = {
        "format": "json",
        "per_page": 1000,
        "page": 1,
        "date": f"{start_year}:{end_year}",
    }

    all_rows = []

    while True:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()

        # handle error / no data
        if not isinstance(data, list) or len(data) < 2 or data[1] is None:
            return pd.DataFrame(columns=["iso3c", "country", "year", indicator])

        meta, rows = data[0], data[1]

        for row in rows:
            iso3 = row.get("countryiso3code")
            year_str = row.get("date")
            if not iso3 or not year_str:
                continue

            all_rows.append({
                "iso3c": iso3,
                "country": (row.get("country") or {}).get("value"),
                "year": int(year_str),
                indicator: row.get("value"),
            })

        if meta.get("page") >= meta.get("pages"):
            break
        params["page"] += 1

    return pd.DataFrame(all_rows)

#choose years
START_YEAR = 2000
END_YEAR   = 2023

#target + predictors (NO CO2) 
INDICATORS = {
    "mmr": "SH.STA.MMRT",              # maternal mortality ratio
    "gdp_pc": "NY.GDP.PCAP.CD",        # GDP per capita
    "health_exp_gdp": "SH.XPD.CHEX.GD.ZS",  # health expenditure (% GDP)
    "fertility": "SP.DYN.TFRT.IN",     # fertility rate
    "skilled_births": "SH.STA.BRTC.ZS",# skilled birth attendance (%)
    "pm25": "EN.ATM.PM25.MC.M3",       # PM2.5 exposure
    "heat_index35_days": "EN.CLC.HEAT.XD",  # heat index >=35C days
    "cooling_degree_days": "EN.CLC.CDDY.XD",# cooling degree days
    "disaster_exposure": "EN.CLC.MDAT.ZS",  # disaster exposure (% pop)
}

#fetch + merge
dfs = []
for col, code in INDICATORS.items():
    temp = fetch_wb(code, START_YEAR, END_YEAR).rename(columns={code: col})
    print(col, code, temp.shape)
    dfs.append(temp)

df = dfs[0]
for temp in dfs[1:]:
    df = df.merge(temp, on=["iso3c", "country", "year"], how="outer")

#convert to numeric
for col in INDICATORS.keys():
    df[col] = pd.to_numeric(df[col], errors="coerce")

#keep rows with target present
df = df.dropna(subset=["mmr"]).sort_values(["iso3c", "year"]).reset_index(drop=True)

#simple imputation for predictors: fill missing by year-median
predictors = [c for c in INDICATORS.keys() if c != "mmr"]
df[predictors] = df.groupby("year")[predictors].transform(lambda s: s.fillna(s.median()))

print("Final dataset shape:", df.shape)
df.head()

mmr SH.STA.MMRT (6264, 4)
gdp_pc NY.GDP.PCAP.CD (6264, 4)
health_exp_gdp SH.XPD.CHEX.GD.ZS (6264, 4)
fertility SP.DYN.TFRT.IN (6264, 4)
skilled_births SH.STA.BRTC.ZS (6264, 4)
pm25 EN.ATM.PM25.MC.M3 (6264, 4)
heat_index35_days EN.CLC.HEAT.XD (4920, 4)
cooling_degree_days EN.CLC.CDDY.XD (4920, 4)
disaster_exposure EN.CLC.MDAT.ZS (6264, 4)
Final dataset shape: (5712, 12)


,iso3c,country,year,mmr,gdp_pc,health_exp_gdp,fertility,skilled_births,pm25,heat_index35_days,cooling_degree_days,disaster_exposure
0,AFE,Africa Eastern and Southern,2000,612.0,706.727261,5.657523,5.549893,40.977528,26.783513,0.0,3061.85,NaN
1,AFE,Africa Eastern and Southern,2001,586.0,625.827815,5.803267,5.501766,98.500000,26.809926,0.0,3255.66,NaN
2,AFE,Africa Eastern and Southern,2002,564.0,630.512328,5.408213,5.449696,98.900000,26.718091,0.0,3474.73,NaN
3,AFE,Africa Eastern and Southern,2003,537.0,815.432296,5.989670,5.407328,98.900000,26.580603,0.0,3506.74,NaN
4,AFE,Africa Eastern and Southern,2004,516.0,989.015464,6.069927,5.381308,98.500000,26.475208,0.0,3467.88,NaN


In [32]:
df.tail()

,iso3c,country,year,mmr,gdp_pc,health_exp_gdp,fertility,skilled_births,pm25,heat_index35_days,cooling_degree_days,disaster_exposure
5707,ZWE,Zimbabwe,2019,388.0,2184.329239,3.232682,3.748,86.00,18.528607,0.21,2998.17,NaN
5708,ZWE,Zimbabwe,2020,380.0,2059.674454,2.954401,3.754,99.25,19.494180,0.11,2640.14,NaN
5709,ZWE,Zimbabwe,2021,446.0,2613.605421,2.671119,3.765,99.35,NaN,NaN,NaN,NaN
5710,ZWE,Zimbabwe,2022,368.0,2536.400502,3.395195,3.767,98.70,NaN,NaN,NaN,NaN
5711,ZWE,Zimbabwe,2023,358.0,2195.224921,2.926016,3.724,NaN,NaN,NaN,NaN,NaN


In [34]:
df.to_csv("country_year_modeling_dataset.csv", index=False)
print("Saved: country_year_modeling_dataset.csv")

Saved: country_year_modeling_dataset.csv


In [36]:
print(df.describe())
print(df.isna().sum())
print(df["year"].min(), df["year"].max())
print(df["mmr"].describe())

              year          mmr         gdp_pc  health_exp_gdp    fertility  \
count  5712.000000  5712.000000    5712.000000     5712.000000  5712.000000   
mean   2011.500000   185.514006   12315.948257        6.164527     2.918973   
std       6.922793   250.643383   20211.618338        2.783244     1.484681   
min    2000.000000     1.000000     109.593814        1.222602     0.721000   
25%    2005.750000    17.000000    1451.379924        4.229262     1.730000   
50%    2011.500000    74.000000    4344.474745        5.419955     2.414000   
75%    2017.250000   258.000000   13028.965520        7.735592     3.913000   
max    2023.000000  1662.000000  256799.788613       27.089685     7.829000   

       skilled_births         pm25  heat_index35_days  cooling_degree_days  \
count     5474.000000  4998.000000        4998.000000          4998.000000   
mean        94.544927    28.290154           5.448285          3225.090508   
std         13.586242    16.526619          17.167828 